<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase2_Compare_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Phase 2 — Compare All Experiments

**เลือก best mask method จาก 3 experiments เพื่อใช้ใน Phase 3**

> 🔗 **ต้องรัน Exp 1, 2, 3 ครบทั้งหมดก่อน** (หรืออย่างน้อยที่ทำได้)

---

## Comparison ทำอะไร?

1. โหลด `results.json` ของทั้ง 3 experiments
2. เปรียบเทียบ IoU + Dice + per-class + per-source
3. เลือก best experiment (highest mean IoU) → mark สำหรับ Phase 3
4. Save summary report สำหรับ Final Report

**⏱ เวลา: ~2 นาที** (ไม่ต้องใช้ GPU)


In [ ]:
# Setup + load results
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'
PHASE2_DIR = f'{PROJECT_ROOT}/phase2'

EXPERIMENTS = {
    'Exp 1: SAM 3':                f'{PHASE2_DIR}/exp1_sam3/results.json',
    'Exp 2: Florence-2 → SAM 3':   f'{PHASE2_DIR}/exp2_florence2_sam3/results.json',
    'Exp 3: DINOv3 + seg head':    f'{PHASE2_DIR}/exp3_dinov3/results.json',
}

# Check which experiments have completed
loaded = {}
for name, path in EXPERIMENTS.items():
    if os.path.exists(path):
        with open(path) as f:
            loaded[name] = json.load(f)
        print(f'[OK]   {name:<32s} → loaded')
    else:
        print(f'[!]    {name:<32s} → not run yet ({path})')

if len(loaded) == 0:
    raise RuntimeError('No experiments completed — รัน Exp 1/2/3 อย่างน้อย 1 ตัวก่อน')
print(f'\nLoaded {len(loaded)}/{len(EXPERIMENTS)} experiments')


## Comparison Table

เปรียบเทียบ overall metrics + per-class + per-source ของทุก experiment ที่รันได้


In [ ]:
# Helper: extract metrics handling different schemas
def get_iou_mean(result):
    if 'aggregate' in result:
        return result['aggregate']['mean_iou']
    elif 'aggregate_pooled' in result:
        return result['aggregate_pooled']['mean_iou']
    return None

def get_iou_std(result):
    if 'aggregate' in result:
        return result['aggregate']['std_iou']
    elif 'aggregate_pooled' in result:
        return result['aggregate_pooled']['std_iou']
    return None

def get_dice_mean(result):
    if 'aggregate' in result:
        return result['aggregate']['mean_dice']
    elif 'aggregate_pooled' in result:
        return result['aggregate_pooled']['mean_dice']
    return None

def get_dice_std(result):
    if 'aggregate' in result:
        return result['aggregate']['std_dice']
    elif 'aggregate_pooled' in result:
        return result['aggregate_pooled']['std_dice']
    return None


# Build summary DataFrame
rows = []
for name, r in loaded.items():
    rows.append({
        'experiment': name,
        'mean_iou':  get_iou_mean(r),
        'std_iou':   get_iou_std(r),
        'mean_dice': get_dice_mean(r),
        'std_dice':  get_dice_std(r),
    })

summary_df = pd.DataFrame(rows).sort_values('mean_iou', ascending=False).reset_index(drop=True)
print('OVERALL COMPARISON (sorted by mean IoU):\n')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

BEST_EXP = summary_df.iloc[0]['experiment']
print(f'\n★ Best experiment: {BEST_EXP}')
print(f'   Mean IoU = {summary_df.iloc[0]["mean_iou"]:.4f}')


In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = summary_df['experiment'].tolist()
colors = ['#1e3a5f', '#c4583a', '#4a7c59'][:len(names)]
x_pos = range(len(names))

# IoU
axes[0].bar(x_pos, summary_df['mean_iou'], yerr=summary_df['std_iou'],
             color=colors, alpha=0.85, capsize=5)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([n.replace(' → ', '\n→ ').replace(' + ', '\n+ ') for n in names],
                         fontsize=9)
axes[0].set_ylabel('Mean IoU')
axes[0].set_title('IoU Comparison (mean ± std)')
axes[0].grid(alpha=0.3, axis='y')
axes[0].axhline(0.5, color='gray', linestyle=':', label='IoU=0.5 (acceptable)')
axes[0].legend()
for i, val in enumerate(summary_df['mean_iou']):
    axes[0].text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# Dice
axes[1].bar(x_pos, summary_df['mean_dice'], yerr=summary_df['std_dice'],
             color=colors, alpha=0.85, capsize=5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([n.replace(' → ', '\n→ ').replace(' + ', '\n+ ') for n in names],
                         fontsize=9)
axes[1].set_ylabel('Mean Dice')
axes[1].set_title('Dice Comparison (mean ± std)')
axes[1].grid(alpha=0.3, axis='y')
for i, val in enumerate(summary_df['mean_dice']):
    axes[1].text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PHASE2_DIR}/comparison_overall.png', dpi=100, bbox_inches='tight')
plt.show()


## Per-Class + Per-Source Drill-down

ดูว่า experiment ไหนเก่งกับ class/source อะไร — อาจมี trade-off (e.g., Exp 1 ดี Fresh แต่ Exp 3 ดี Spoiled)


In [ ]:
# Per-class IoU comparison
class_rows = []
for name, r in loaded.items():
    if 'per_class' in r:
        for cls, metrics in r['per_class'].items():
            class_rows.append({
                'experiment': name, 'class': cls, 'iou': metrics['iou_mean'], 'dice': metrics['dice_mean']
            })

if class_rows:
    pc_df = pd.DataFrame(class_rows)
    pivot_iou = pc_df.pivot(index='experiment', columns='class', values='iou')
    print('Per-class IoU:\n')
    print(pivot_iou.to_string(float_format=lambda x: f'{x:.3f}'))

    # Heatmap
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot_iou.values, cmap='YlGnBu', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot_iou.columns)))
    ax.set_xticklabels(pivot_iou.columns)
    ax.set_yticks(range(len(pivot_iou.index)))
    ax.set_yticklabels(pivot_iou.index)
    for i in range(pivot_iou.shape[0]):
        for j in range(pivot_iou.shape[1]):
            val = pivot_iou.values[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', color=color, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title('Per-Class IoU Heatmap')
    plt.tight_layout()
    plt.savefig(f'{PHASE2_DIR}/comparison_per_class.png', dpi=100, bbox_inches='tight')
    plt.show()


In [ ]:
# Per-source IoU comparison (packaged vs unpackaged)
source_rows = []
for name, r in loaded.items():
    if 'per_source' in r:
        for src, metrics in r['per_source'].items():
            source_rows.append({
                'experiment': name, 'source': src, 'iou': metrics['iou_mean'], 'dice': metrics['dice_mean']
            })

if source_rows:
    ps_df = pd.DataFrame(source_rows)
    pivot_src = ps_df.pivot(index='experiment', columns='source', values='iou')
    print('Per-source IoU:\n')
    print(pivot_src.to_string(float_format=lambda x: f'{x:.3f}'))

    # Bar chart grouped
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(pivot_src.index))
    width = 0.35
    for i, src in enumerate(pivot_src.columns):
        ax.bar(x + i * width, pivot_src[src].values, width, label=src, alpha=0.85)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(pivot_src.index, fontsize=9)
    ax.set_ylabel('Mean IoU')
    ax.set_title('Per-Source IoU (Packaged vs Unpackaged)')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{PHASE2_DIR}/comparison_per_source.png', dpi=100, bbox_inches='tight')
    plt.show()


## Decision: Best Method for Phase 3

เลือก experiment ที่:
1. **Mean IoU สูงสุด** (primary criterion)
2. **Variance ต่ำ** (consistent across samples)
3. **ทำงานบน packaged samples ได้ดี** (เพราะเป็น use case จริง)


In [ ]:
# Save decision report for Phase 3
decision = {
    'best_experiment': BEST_EXP,
    'reason': 'Highest mean IoU on Thai test set',
    'overall_summary': summary_df.to_dict('records'),
    'per_class_breakdown': pc_df.to_dict('records') if class_rows else None,
    'per_source_breakdown': ps_df.to_dict('records') if source_rows else None,
}

decision_path = f'{PHASE2_DIR}/best_experiment_decision.json'
with open(decision_path, 'w') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)

# Determine which directory contains the best mask predictions
exp_to_dir = {
    'Exp 1: SAM 3':                f'{PHASE2_DIR}/exp1_sam3/predicted_masks',
    'Exp 2: Florence-2 → SAM 3':   f'{PHASE2_DIR}/exp2_florence2_sam3/predicted_masks',
    'Exp 3: DINOv3 + seg head':    f'{PHASE2_DIR}/exp3_dinov3/predicted_masks',
}
BEST_MASKS_DIR = exp_to_dir[BEST_EXP]
print(f'[OK] Decision saved: {decision_path}')
print(f'\nPhase 3 will use masks from: {BEST_MASKS_DIR}')


In [ ]:
# Final summary
print('=' * 65)
print('PHASE 2 — COMPLETE')
print('=' * 65)
print(f'\nExperiments completed: {len(loaded)}/{len(EXPERIMENTS)}\n')
print('Ranking by mean IoU on Thai test set:\n')
for i, row in summary_df.iterrows():
    medal = ['🥇', '🥈', '🥉'][i] if i < 3 else '  '
    print(f'  {medal} {row["experiment"]:<32s}'
          f' IoU={row["mean_iou"]:.4f}±{row["std_iou"]:.4f}'
          f' Dice={row["mean_dice"]:.4f}±{row["std_dice"]:.4f}')

print(f'\n★ BEST: {BEST_EXP}')
print(f'\nFiles created:')
print(f'  {PHASE2_DIR}/comparison_overall.png')
print(f'  {PHASE2_DIR}/comparison_per_class.png')
print(f'  {PHASE2_DIR}/comparison_per_source.png')
print(f'  {PHASE2_DIR}/best_experiment_decision.json')
print(f'\nNext: Phase 3 will load masks from {BEST_MASKS_DIR}')
print(f'      → run pipeline with 3 classifiers (EfficientNet, Swin-T, DINOv3)')
